
# CS425 Diss-Poem Generator — Google Colab Training Notebook

This notebook implements a **three-source training/generation pipeline**:

1. **Sonnet dataset → poetic pretraining**
   - Teaches general poetic phrasing, line transitions, and literary structure.
2. **Diss dataset → style fine-tuning**
   - Adapts the same model toward diss-style vocabulary, direct address, punchlines, and confrontational phrasing.
3. **CSUdict / CMUdict-style pronunciation dictionary → rhyme and syllable control**
   - Used **after neural training** as a phonetic knowledge source rather than as ordinary language-model training text.

### Methodology

```text
SONNETS
   │
   ▼
Stage 1: GRU language-model pretraining
   │
   ▼
Poetry-aware model
   │
   ▼
DISS DATASET
   │
   ▼
Stage 2: diss-style fine-tuning
   │
   ▼
Diss-poem neural generator
   │
   ├── candidate line 1
   ├── candidate line 2
   ├── candidate line 3
   └── ...
          │
          ▼
CSUdict / CMUdict phonetic scoring
   ├── rhyme score
   └── syllable / meter score
          │
          ▼
Best candidate selected
```

The model used here is a **small PyTorch GRU language model**, designed to be trainable on Google Colab without relying on an LLM API.

> **Before running:** In Colab, choose **Runtime → Change runtime type → GPU**.



## Part 0 — Install dependencies

### What this block does
Installs the lightweight libraries used by the notebook.

### Methodology
Google Colab already includes PyTorch in most runtimes, so the notebook **does not reinstall PyTorch**. This avoids accidentally replacing Colab's CUDA-compatible PyTorch build.

We install:
- `nltk` — optional CMU pronunciation dictionary fallback
- `pandas` — loading CSV datasets
- `matplotlib` — training curves
- `tqdm` — progress bars


In [ ]:

"""
TRAINING PROCESS — PART 0: DEPENDENCY INSTALLATION

Purpose:
    Install supporting packages needed by the Colab notebook.

Methodology:
    Keep Colab's preinstalled PyTorch/CUDA build and install only
    the additional lightweight dependencies.
"""

!pip -q install nltk pandas matplotlib tqdm



## Part 1 — Imports, reproducibility, and GPU check

### What this block does
Imports libraries, fixes random seeds, checks whether CUDA is available, and creates the working directories.

### Methodology
A fixed seed makes model initialization and dataset splitting more reproducible. Exact equality between GPU runs is not guaranteed, but this significantly reduces accidental variation.


In [ ]:

"""
TRAINING PROCESS — PART 1: ENVIRONMENT SETUP

Purpose:
    Import libraries, set random seeds, select CPU/GPU,
    and create output/checkpoint directories.

Methodology:
    Use a fixed seed for reproducibility and automatically select
    CUDA when a Colab GPU runtime is available.
"""

from pathlib import Path
from collections import Counter
import csv
import json
import math
import random
import re
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PROJECT_DIR = Path("/content/diss_poem_project")
DATA_DIR = PROJECT_DIR / "data"
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"
OUTPUT_DIR = PROJECT_DIR / "outputs"

for directory in [DATA_DIR, CHECKPOINT_DIR, OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("PyTorch version:", torch.__version__)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU not detected. Training will still work, but more slowly.")



## Part 2 — Configuration

### What this block does
Defines the dataset paths, dataset-column names, model size, and training hyperparameters.

### Methodology
All important settings are centralized so experiments can be repeated consistently. For the project report, this cell also provides a convenient record of the chosen hyperparameters.

### Expected dataset format

**Sonnet file**: `sonnets.txt`
- Sonnets/poems separated by one or more blank lines, or consecutive 14-line sonnets.

**Diss file**: `diss.csv`
- One training sample per row.
- Default text column name: `text`.

**Pronunciation dictionary**: `csudict.dict`, `cmudict.dict`, or similar
- Expected CMU-style format such as:

```text
CAT  K AE1 T
HAT  HH AE1 T
ATTACK  AH0 T AE1 K
```

If no custom pronunciation dictionary is uploaded, the notebook can fall back to NLTK's CMUdict.


In [ ]:

"""
TRAINING PROCESS — PART 2: CONFIGURATION

Purpose:
    Centralize file paths, preprocessing choices, model architecture,
    and training hyperparameters.

Methodology:
    Keeping these settings in one cell makes experimentation and
    reporting easier and reduces hidden changes between runs.
"""

# ------------------------------------------------------------------
# Dataset paths
# ------------------------------------------------------------------

SONNET_PATH = DATA_DIR / "sonnets.txt"
DISS_PATH = DATA_DIR / "diss.csv"
PRON_DICT_PATH = DATA_DIR / "csudict.dict"
CLEANED_DIR = DATA_DIR / "cleaned"
SONNET_CLEANED_PATH = CLEANED_DIR / "sonnets_cleaned.txt"
DISS_CLEANED_PATH = CLEANED_DIR / "diss_cleaned.csv"

# Change this if your diss CSV uses another column name.
DISS_TEXT_COLUMN = "text"

# If the custom pronunciation dictionary is absent, use NLTK CMUdict.
USE_NLTK_CMUDICT_IF_MISSING = True

# ------------------------------------------------------------------
# Text preprocessing
# ------------------------------------------------------------------

LOWERCASE = True
MIN_TOKEN_FREQUENCY = 1

# ------------------------------------------------------------------
# Sequence/model settings
# ------------------------------------------------------------------

SEQUENCE_LENGTH = 64
BATCH_SIZE = 64

EMBEDDING_DIM = 128
HIDDEN_DIM = 256
NUM_LAYERS = 2
DROPOUT = 0.25

# ------------------------------------------------------------------
# Training settings
# ------------------------------------------------------------------

VALIDATION_SPLIT = 0.10

SONNET_EPOCHS = 5
DISS_EPOCHS = 5

SONNET_LEARNING_RATE = 3e-3
DISS_LEARNING_RATE = 1e-3

WEIGHT_DECAY = 1e-4
GRADIENT_CLIP_NORM = 1.0

# ------------------------------------------------------------------
# Generation settings
# ------------------------------------------------------------------

GENERATION_TEMPERATURE = 0.9
TOP_K = 30
CANDIDATES_PER_LINE = 8
TARGET_SYLLABLES = 10
MAX_TOKENS_PER_LINE = 20
MAX_CONTEXT_TOKENS = 256

print("Configuration loaded.")



## Part 3 — Upload datasets to Colab

### What this block does
Lets you upload the sonnet, diss, and optional pronunciation-dictionary files directly from your computer.

### Methodology
The notebook renames recognized files into the standard names expected by the configuration cell.

Cleaned files named `*_cleaned.txt` or `*_cleaned.csv` are placed in `/content/diss_poem_project/data/cleaned/` and take priority over raw files.

If your files already exist in `/content/diss_poem_project/data/` (or its `cleaned/` folder), you can skip this cell.

> For larger datasets, you may instead mount Google Drive and update the paths in **Part 2**.


In [ ]:

"""
TRAINING PROCESS — PART 3: DATASET UPLOAD

Purpose:
    Upload local dataset files into the Colab runtime.

Methodology:
    Use Colab's upload interface, then copy recognized file types
    into the project data directory.

Expected:
    - a sonnet TXT file
    - a diss CSV file
    - optionally a CMU/CSU-style pronunciation dictionary
"""

from google.colab import files

uploaded = files.upload()

for filename, data in uploaded.items():
    source_name = filename.lower()

    if source_name.endswith(".txt") and "sonnet" in source_name:
        destination = (
            SONNET_CLEANED_PATH if source_name.endswith("_cleaned.txt")
            else SONNET_PATH
        )

    elif source_name.endswith(".csv") and "diss" in source_name:
        destination = (
            DISS_CLEANED_PATH if source_name.endswith("_cleaned.csv")
            else DISS_PATH
        )

    elif (
        source_name.endswith(".dict")
        or "cmudict" in source_name
        or "csudict" in source_name
    ):
        destination = PRON_DICT_PATH

    else:
        # Unrecognized files are still saved so you can manually
        # update the configured path if needed.
        destination = DATA_DIR / filename

    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(data)
    print(f"Saved {filename} -> {destination}")



## Part 4 — Load and clean the datasets

### What this block does
Loads both text corpora and applies basic normalization. The first run saves `sonnets_cleaned.txt` and `diss_cleaned.csv` in `data/cleaned/`. Later runs load each cleaned file directly when it exists, even if the raw file is absent.

### Methodology
The sonnet and diss datasets are deliberately kept **separate** because they are used in different training stages.
Sonnet files without blank-line separators are grouped into 14-line samples.

Line breaks are preserved because line structure matters for poetry. Excess whitespace is removed, and optional lowercasing reduces vocabulary size for a relatively small model.

Delete an individual `_cleaned` file when its raw dataset or preprocessing settings change and you want to regenerate it.


In [ ]:

"""
TRAINING PROCESS — PART 4: DATA LOADING AND CLEANING

Purpose:
    Read the sonnet and diss corpora into separate Python lists.

Methodology:
    Preserve poem line boundaries while removing unnecessary
    whitespace. Keep the two corpora separate for staged training.
"""

def clean_text(text: str, lowercase: bool = True) -> str:
    """
    Normalize one poem/diss sample while preserving line boundaries.
    """
    text = str(text).replace("\r\n", "\n").replace("\r", "\n")

    cleaned_lines = []
    for line in text.split("\n"):
        line = re.sub(r"\s+", " ", line).strip()
        if line:
            cleaned_lines.append(line)

    text = "\n".join(cleaned_lines)

    if lowercase:
        text = text.lower()

    return text


def split_sonnet_samples(raw_text: str):
    """
    Use blank-line poem boundaries, or 14-line sonnets if none exist.
    """
    samples = [sample for sample in re.split(r"\n\s*\n", raw_text) if sample.strip()]

    if len(samples) == 1:
        lines = [line for line in raw_text.splitlines() if line.strip()]
        if len(lines) >= 28:
            print("No blank-line sonnet separators found; grouping every 14 lines.")
            if len(lines) % 14:
                print("WARNING: Sonnet line count is not divisible by 14; check poem boundaries.")
            return ["\n".join(lines[index:index + 14]) for index in range(0, len(lines), 14)]

    return samples


def load_sonnet_txt(path: Path, cleaned_path: Path):
    """
    Load cached poems when available; otherwise clean and save the raw poems.
    """
    if cleaned_path.exists():
        source_path = cleaned_path
        print(f"Using cleaned sonnet dataset: {cleaned_path}")
    else:
        if not path.exists():
            raise FileNotFoundError(
                f"Sonnet file not found: {path}\n"
                f"No cleaned dataset found at: {cleaned_path}\n"
                "Upload a file in Part 3 or change SONNET_PATH."
            )
        source_path = path

    raw_text = source_path.read_text(encoding="utf-8", errors="ignore")
    samples = split_sonnet_samples(raw_text)

    if source_path == cleaned_path:
        return samples

    samples = [clean_text(sample, lowercase=LOWERCASE) for sample in samples]
    cleaned_path.parent.mkdir(parents=True, exist_ok=True)
    cleaned_path.write_text("\n\n".join(samples), encoding="utf-8")
    print(f"Saved cleaned sonnet dataset: {cleaned_path}")
    return samples


def load_diss_csv(path: Path, text_column: str, cleaned_path: Path):
    """
    Load cached rows when available; otherwise clean and save raw CSV rows.
    """
    if cleaned_path.exists():
        source_path = cleaned_path
        print(f"Using cleaned diss dataset: {cleaned_path}")
    else:
        if not path.exists():
            raise FileNotFoundError(
                f"Diss CSV not found: {path}\n"
                f"No cleaned dataset found at: {cleaned_path}\n"
                "Upload a file in Part 3 or change DISS_PATH."
            )
        source_path = path

    df = pd.read_csv(source_path)

    if text_column not in df.columns:
        raise ValueError(
            f"Column '{text_column}' not found in {source_path}.\n"
            f"Available columns: {list(df.columns)}"
        )

    samples = [str(value) for value in df[text_column].dropna() if str(value)]

    if source_path == cleaned_path:
        return samples

    samples = [clean_text(sample, lowercase=LOWERCASE) for sample in samples]
    samples = [sample for sample in samples if sample]
    cleaned_path.parent.mkdir(parents=True, exist_ok=True)
    with cleaned_path.open("w", encoding="utf-8", newline="") as output:
        writer = csv.writer(output)
        writer.writerow([text_column])
        writer.writerows([sample] for sample in samples)
    print(f"Saved cleaned diss dataset: {cleaned_path}")
    return samples


sonnet_samples = load_sonnet_txt(SONNET_PATH, SONNET_CLEANED_PATH)
diss_samples = load_diss_csv(DISS_PATH, DISS_TEXT_COLUMN, DISS_CLEANED_PATH)

print(f"Sonnet samples: {len(sonnet_samples):,}")
print(f"Diss samples:    {len(diss_samples):,}")

print("\n--- Example sonnet sample ---")
print(sonnet_samples[0][:600])

print("\n--- Example diss sample ---")
print(diss_samples[0][:600])



## Part 5 — Split train and validation data at the **sample level**

### What this block does
Separates complete poems/diss samples into training and validation subsets **before** creating overlapping token windows.

### Methodology
This avoids a common form of data leakage. If we created overlapping chunks first and then randomly split the chunks, nearly identical text windows from the same poem could appear in both training and validation data.

Instead:

```text
Whole poems/samples
       │
       ├── training samples
       └── validation samples
              │
              ▼
        token windows created
```


In [ ]:

"""
TRAINING PROCESS — PART 5: SAMPLE-LEVEL TRAIN/VALIDATION SPLIT

Purpose:
    Split complete source samples before token chunking.

Methodology:
    Prevent overlapping windows from the same original sample from
    leaking into both training and validation sets.
"""

def split_samples(samples, validation_split=0.10, seed=42):
    """
    Deterministically shuffle and split a list of complete samples.
    """
    samples = list(samples)

    rng = random.Random(seed)
    rng.shuffle(samples)

    if len(samples) < 2:
        raise ValueError("Need at least 2 samples for train/validation splitting.")

    validation_size = max(1, int(len(samples) * validation_split))
    validation_size = min(validation_size, len(samples) - 1)

    validation_samples = samples[:validation_size]
    training_samples = samples[validation_size:]

    return training_samples, validation_samples


sonnet_train_samples, sonnet_val_samples = split_samples(
    sonnet_samples,
    validation_split=VALIDATION_SPLIT,
    seed=SEED,
)

diss_train_samples, diss_val_samples = split_samples(
    diss_samples,
    validation_split=VALIDATION_SPLIT,
    seed=SEED,
)

print("Sonnet train / val:", len(sonnet_train_samples), "/", len(sonnet_val_samples))
print("Diss train / val:   ", len(diss_train_samples), "/", len(diss_val_samples))



## Part 6 — Build the tokenizer and shared vocabulary

### What this block does
Creates a simple word-level tokenizer and builds a **shared vocabulary** from the sonnet and diss corpora.

### Methodology
The vocabulary is shared because the same neural network is used across both stages.

Special tokens:
- `<PAD>` — padding
- `<UNK>` — unknown token
- `<BOS>` — beginning of sequence
- `<EOS>` — end of sequence
- `<NL>` — poem line break
- `<SONNET>` — identifies sonnet-style input
- `<DISS>` — identifies diss-style input

The explicit `<NL>` token helps the model learn where poetic lines end.


In [ ]:

"""
TRAINING PROCESS — PART 6: TOKENIZATION AND VOCABULARY

Purpose:
    Convert raw text to tokens and create a shared vocabulary.

Methodology:
    Use explicit newline/style tokens so a small word-level GRU can
    learn both text content and high-level corpus identity.
"""

SPECIAL_TOKENS = [
    "<PAD>",
    "<UNK>",
    "<BOS>",
    "<EOS>",
    "<NL>",
    "<SONNET>",
    "<DISS>",
]

TOKEN_PATTERN = re.compile(
    r"<[^>]+>"
    r"|[A-Za-z]+(?:'[A-Za-z]+)?"
    r"|[0-9]+"
    r"|[^\w\s]"
)


class WordTokenizer:
    """
    Lightweight word-level tokenizer for the project model.
    """

    def __init__(self, min_frequency=1):
        self.min_frequency = min_frequency
        self.token_to_id = {}
        self.id_to_token = {}

    @staticmethod
    def tokenize(text):
        text = text.replace("\n", " <NL> ")
        return TOKEN_PATTERN.findall(text)

    def fit(self, texts):
        counts = Counter()

        for text in texts:
            counts.update(self.tokenize(text))

        vocabulary = list(SPECIAL_TOKENS)

        for token, frequency in counts.items():
            if (
                frequency >= self.min_frequency
                and token not in SPECIAL_TOKENS
            ):
                vocabulary.append(token)

        self.token_to_id = {
            token: index
            for index, token in enumerate(vocabulary)
        }

        self.id_to_token = {
            index: token
            for token, index in self.token_to_id.items()
        }

    def encode(
        self,
        text,
        style_token=None,
        add_bos=True,
        add_eos=True
    ):
        ids = []

        if add_bos:
            ids.append(self.token_to_id["<BOS>"])

        if style_token is not None:
            ids.append(self.token_to_id[style_token])

        unk_id = self.token_to_id["<UNK>"]

        for token in self.tokenize(text):
            ids.append(self.token_to_id.get(token, unk_id))

        if add_eos:
            ids.append(self.token_to_id["<EOS>"])

        return ids

    def decode(self, ids):
        tokens = [self.id_to_token[int(index)] for index in ids]

        output = []

        for token in tokens:
            if token in {
                "<PAD>",
                "<BOS>",
                "<EOS>",
                "<SONNET>",
                "<DISS>",
            }:
                continue

            if token == "<NL>":
                output.append("\n")
            else:
                output.append(token)

        text = " ".join(output)

        # Remove spaces before punctuation.
        text = re.sub(r"\s+([,.!?;:])", r"\1", text)
        text = re.sub(r"\s*\n\s*", "\n", text)

        return text.strip()

    def __len__(self):
        return len(self.token_to_id)


tokenizer = WordTokenizer(min_frequency=MIN_TOKEN_FREQUENCY)

# Vocabulary is built from the complete available corpora so Stage 2
# does not introduce a large number of unknown tokens.
tokenizer.fit(sonnet_samples + diss_samples)

print("Vocabulary size:", len(tokenizer))
print("Example tokens:", tokenizer.tokenize("you talk big,\nbut your rhyme falls flat."))

# Save the vocabulary for reproducibility.
with open(OUTPUT_DIR / "vocabulary.json", "w", encoding="utf-8") as f:
    json.dump(tokenizer.token_to_id, f, ensure_ascii=False, indent=2)



## Part 7 — Convert text into next-token prediction examples

### What this block does
Turns each poem/sample into fixed-length training windows.

### Methodology
The network is trained autoregressively: given the current tokens, it predicts the next token.

Example:

```text
Input:   <BOS> <DISS> you talk big but
Target:  <DISS> you talk big but your
```

Long samples are divided into overlapping windows to create more training examples while preserving local context.


In [ ]:

"""
TRAINING PROCESS — PART 7: AUTOREGRESSIVE DATASET CREATION

Purpose:
    Convert samples into fixed-length (input, target) pairs.

Methodology:
    Use next-token prediction with overlapping windows.
    Training/validation separation already happened at the original
    sample level in Part 5.
"""

class CorpusLMDataset(Dataset):
    """
    Fixed-length autoregressive language-model dataset.
    """

    def __init__(
        self,
        samples,
        tokenizer,
        style_token,
        sequence_length=64,
        stride=None
    ):
        self.examples = []

        if stride is None:
            stride = max(1, (sequence_length - 2) // 2)

        bos_id = tokenizer.token_to_id["<BOS>"]
        eos_id = tokenizer.token_to_id["<EOS>"]
        pad_id = tokenizer.token_to_id["<PAD>"]
        style_id = tokenizer.token_to_id[style_token]

        # Two positions are reserved for BOS and the style token.
        max_content_length = sequence_length - 2

        for sample in samples:
            content = tokenizer.encode(
                sample,
                add_bos=False,
                add_eos=False
            )

            if not content:
                continue

            for start in range(0, len(content), stride):
                chunk = content[start:start + max_content_length]

                if not chunk:
                    continue

                sequence = [bos_id, style_id, *chunk]

                # Add EOS when the current chunk reaches the end
                # of the original source sample.
                if start + max_content_length >= len(content):
                    sequence.append(eos_id)

                # Need sequence_length + 1 tokens so that shifting
                # produces an input and a same-length target.
                desired_length = sequence_length + 1

                if len(sequence) < desired_length:
                    sequence += [pad_id] * (desired_length - len(sequence))
                else:
                    sequence = sequence[:desired_length]

                x = torch.tensor(sequence[:-1], dtype=torch.long)
                y = torch.tensor(sequence[1:], dtype=torch.long)

                self.examples.append((x, y))

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        return self.examples[index]


sonnet_train_dataset = CorpusLMDataset(
    sonnet_train_samples,
    tokenizer,
    style_token="<SONNET>",
    sequence_length=SEQUENCE_LENGTH
)

sonnet_val_dataset = CorpusLMDataset(
    sonnet_val_samples,
    tokenizer,
    style_token="<SONNET>",
    sequence_length=SEQUENCE_LENGTH
)

diss_train_dataset = CorpusLMDataset(
    diss_train_samples,
    tokenizer,
    style_token="<DISS>",
    sequence_length=SEQUENCE_LENGTH
)

diss_val_dataset = CorpusLMDataset(
    diss_val_samples,
    tokenizer,
    style_token="<DISS>",
    sequence_length=SEQUENCE_LENGTH
)

print("Sonnet train sequences:", len(sonnet_train_dataset))
print("Sonnet val sequences:  ", len(sonnet_val_dataset))
print("Diss train sequences:   ", len(diss_train_dataset))
print("Diss val sequences:     ", len(diss_val_dataset))



## Part 8 — Define the GRU language model

### What this block does
Defines the neural network that learns both poetry and diss-style text.

### Methodology

```text
Token IDs
   ↓
Embedding layer
   ↓
Stacked GRU
   ↓
Linear output layer
   ↓
Probability distribution over the next token
```

A GRU is used because it is a compact recurrent architecture that is practical to train from scratch on Colab. It also makes the staged-transfer-learning methodology easy to explain.


In [ ]:

"""
TRAINING PROCESS — PART 8: MODEL ARCHITECTURE

Purpose:
    Define a compact word-level GRU language model.

Methodology:
    Learn token embeddings, process sequential context with stacked
    GRUs, and project each hidden state to vocabulary logits.
"""

class DissPoemGRU(nn.Module):
    """
    Small recurrent language model for poetry/diss generation.
    """

    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_dim,
        num_layers,
        dropout,
        pad_id
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=pad_id
        )

        self.gru = nn.GRU(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True
        )

        self.output = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_ids, hidden=None):
        embedded = self.embedding(input_ids)
        sequence_output, hidden = self.gru(embedded, hidden)
        logits = self.output(sequence_output)

        return logits, hidden


model = DissPoemGRU(
    vocab_size=len(tokenizer),
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    pad_id=tokenizer.token_to_id["<PAD>"]
).to(DEVICE)

parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(model)
print(f"\nTrainable parameters: {parameter_count:,}")



## Part 9 — Training and validation utilities

### What this block does
Defines reusable functions for:
- mini-batch training
- validation
- perplexity calculation
- checkpoint saving
- early restoration of the best epoch
- plotting loss curves

### Methodology
The loss is **cross-entropy over the next token**. Padding positions are ignored.

The best checkpoint is selected by **validation loss**, not training loss, to reduce the risk of keeping an overfit final epoch.


In [ ]:

"""
TRAINING PROCESS — PART 9: TRAINING UTILITIES

Purpose:
    Provide reusable training and validation loops.

Methodology:
    Optimize next-token cross-entropy with AdamW, ignore padding,
    clip gradients, and save the checkpoint with the best validation
    loss.
"""

def make_loader(dataset, shuffle):
    """
    Create a DataLoader with the configured batch size.
    """
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )


def run_training_epoch(model, dataloader, optimizer, pad_id):
    """
    Train for one complete epoch.
    """
    model.train()
    total_loss = 0.0

    progress = tqdm(dataloader, leave=False)

    for x, y in progress:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits, _ = model(x)

        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            y.reshape(-1),
            ignore_index=pad_id
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=GRADIENT_CLIP_NORM
        )

        optimizer.step()

        total_loss += loss.item()
        progress.set_postfix(loss=f"{loss.item():.3f}")

    return total_loss / max(1, len(dataloader))


@torch.no_grad()
def run_validation(model, dataloader, pad_id):
    """
    Measure average validation loss without updating parameters.
    """
    model.eval()
    total_loss = 0.0

    for x, y in dataloader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        logits, _ = model(x)

        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            y.reshape(-1),
            ignore_index=pad_id
        )

        total_loss += loss.item()

    return total_loss / max(1, len(dataloader))


def fit_stage(
    model,
    train_dataset,
    val_dataset,
    epochs,
    learning_rate,
    stage_name
):
    """
    Train one full stage and reload the best checkpoint.
    """
    train_loader = make_loader(train_dataset, shuffle=True)
    val_loader = make_loader(val_dataset, shuffle=False)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=WEIGHT_DECAY
    )

    pad_id = tokenizer.token_to_id["<PAD>"]
    best_val_loss = float("inf")
    best_checkpoint = CHECKPOINT_DIR / f"{stage_name}_best.pt"

    history = {
        "train_loss": [],
        "val_loss": [],
        "val_perplexity": []
    }

    for epoch in range(1, epochs + 1):
        train_loss = run_training_epoch(
            model,
            train_loader,
            optimizer,
            pad_id
        )

        val_loss = run_validation(
            model,
            val_loader,
            pad_id
        )

        val_perplexity = math.exp(min(val_loss, 20))

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_perplexity"].append(val_perplexity)

        print(
            f"[{stage_name}] Epoch {epoch:02d}/{epochs} | "
            f"train loss={train_loss:.4f} | "
            f"val loss={val_loss:.4f} | "
            f"val perplexity={val_perplexity:.2f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss

            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "stage_name": stage_name,
                    "epoch": epoch,
                    "validation_loss": val_loss,
                    "config": {
                        "embedding_dim": EMBEDDING_DIM,
                        "hidden_dim": HIDDEN_DIM,
                        "num_layers": NUM_LAYERS,
                        "dropout": DROPOUT,
                        "sequence_length": SEQUENCE_LENGTH,
                    },
                },
                best_checkpoint
            )

    checkpoint = torch.load(best_checkpoint, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])

    print(
        f"Restored best {stage_name} checkpoint "
        f"(epoch {checkpoint['epoch']}, val loss={checkpoint['validation_loss']:.4f})."
    )

    return history


def plot_history(history, title):
    """
    Plot training and validation loss using Matplotlib defaults.
    """
    epochs = range(1, len(history["train_loss"]) + 1)

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["train_loss"], marker="o", label="Training loss")
    plt.plot(epochs, history["val_loss"], marker="o", label="Validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("Cross-entropy loss")
    plt.title(title)
    plt.legend()
    plt.grid(alpha=0.25)
    plt.show()



# Stage 1 — Sonnet pretraining

## Part 10 — Train first on sonnets

### What this block does
Trains the GRU on the sonnet corpus.

### Methodology
This is a form of **domain pretraining / transfer learning**. The model first learns general poetic patterns before seeing the more specialized diss data.

Expected learning includes:
- poetic word sequences
- line transitions
- literary phrasing
- punctuation and line-break patterns

The model is **not reset** after this stage; the learned weights become the starting point for diss fine-tuning.


In [ ]:

"""
TRAINING PROCESS — PART 10: STAGE 1 — SONNET PRETRAINING

Purpose:
    Train the base GRU on poetic text before specialization.

Methodology:
    Use sonnet next-token prediction to initialize the neural weights
    with poetry-oriented language patterns.
"""

print("=" * 70)
print("STAGE 1: SONNET PRETRAINING")
print("=" * 70)

sonnet_history = fit_stage(
    model=model,
    train_dataset=sonnet_train_dataset,
    val_dataset=sonnet_val_dataset,
    epochs=SONNET_EPOCHS,
    learning_rate=SONNET_LEARNING_RATE,
    stage_name="sonnet"
)

plot_history(
    sonnet_history,
    "Stage 1 — Sonnet Pretraining"
)



# Stage 2 — Diss fine-tuning

## Part 11 — Fine-tune the same model on the diss dataset

### What this block does
Continues training from the best sonnet checkpoint using diss text.

### Methodology
The diss stage uses a **smaller learning rate** than sonnet pretraining so the model can adapt to the new style without immediately overwriting all previously learned poetic structure.

Expected specialization:
- diss vocabulary
- direct-address phrasing
- setup/punchline patterns
- sharper rhetorical language


In [ ]:

"""
TRAINING PROCESS — PART 11: STAGE 2 — DISS FINE-TUNING

Purpose:
    Specialize the poetry-aware model on diss-style text.

Methodology:
    Continue training the same network with a lower learning rate.
    This preserves useful poetic features while adapting its output
    distribution to the diss corpus.
"""

print("=" * 70)
print("STAGE 2: DISS FINE-TUNING")
print("=" * 70)

diss_history = fit_stage(
    model=model,
    train_dataset=diss_train_dataset,
    val_dataset=diss_val_dataset,
    epochs=DISS_EPOCHS,
    learning_rate=DISS_LEARNING_RATE,
    stage_name="diss"
)

plot_history(
    diss_history,
    "Stage 2 — Diss Fine-Tuning"
)

FINAL_MODEL_PATH = CHECKPOINT_DIR / "diss_poem_final.pt"

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "token_to_id": tokenizer.token_to_id,
        "config": {
            "embedding_dim": EMBEDDING_DIM,
            "hidden_dim": HIDDEN_DIM,
            "num_layers": NUM_LAYERS,
            "dropout": DROPOUT,
            "sequence_length": SEQUENCE_LENGTH,
        },
    },
    FINAL_MODEL_PATH
)

print("Final model saved to:", FINAL_MODEL_PATH)



# Stage 3 — Pronunciation-based rhyme and meter control

## Part 12 — Load CSUdict / CMUdict pronunciation data

### What this block does
Loads a custom CMU-style pronunciation dictionary if provided. Otherwise, it downloads NLTK's CMUdict as a fallback.

### Methodology
The pronunciation dictionary is **not treated as language-model training text**. It acts as a structured phonetic knowledge base after training.

It provides:

```text
word → phonemes → rhyme ending + syllable estimate
```

This keeps the responsibilities separate:
- neural model → content and style
- pronunciation dictionary → phonetic constraints


In [ ]:

"""
TRAINING PROCESS — PART 12: PRONUNCIATION DICTIONARY

Purpose:
    Load phoneme sequences for rhyme and syllable analysis.

Methodology:
    Prefer the project's custom CSUdict/CMUdict-style file.
    If it is missing and fallback is enabled, download NLTK CMUdict.
"""

def load_custom_pronunciation_dictionary(path: Path):
    """
    Parse CMU-style dictionary lines:
        WORD  PHONEME PHONEME ...
    """
    pronunciations = {}

    with path.open("r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()

            if not line or line.startswith(";;;"):
                continue

            parts = line.split()

            if len(parts) < 2:
                continue

            raw_word = parts[0]
            word = re.sub(r"\(\d+\)$", "", raw_word).lower()
            phones = parts[1:]

            pronunciations.setdefault(word, []).append(phones)

    return pronunciations


if PRON_DICT_PATH.exists():
    pronunciations = load_custom_pronunciation_dictionary(PRON_DICT_PATH)
    pronunciation_source = str(PRON_DICT_PATH)

elif USE_NLTK_CMUDICT_IF_MISSING:
    import nltk

    nltk.download("cmudict", quiet=True)
    from nltk.corpus import cmudict

    pronunciations = cmudict.dict()
    pronunciation_source = "NLTK CMUdict fallback"

else:
    raise FileNotFoundError(
        f"No pronunciation dictionary found at {PRON_DICT_PATH}"
    )

print("Pronunciation source:", pronunciation_source)
print("Words with pronunciations:", len(pronunciations))



## Part 13 — Convert pronunciations into rhyme and syllable features

### What this block does
Defines functions to:
- extract rhyme keys from ARPAbet-like phoneme sequences
- test whether two words rhyme
- estimate syllables in a word
- estimate syllables in a generated line

### Methodology
A rhyme key is taken from the **last stressed vowel onward**. Two words receive a rhyme match if any known pronunciation shares the same rhyme key.

Syllables are approximated by counting phonemes marked with ARPAbet stress digits (`0`, `1`, or `2`).


In [ ]:

"""
TRAINING PROCESS — PART 13: PHONETIC FEATURE EXTRACTION

Purpose:
    Produce rhyme keys and syllable counts from pronunciation data.

Methodology:
    - Rhyme key: final stressed vowel + following phonemes
    - Syllables: count phonemes carrying stress digits
"""

def pronunciations_for(word):
    """
    Return all known pronunciations for a normalized word.
    """
    if not word:
        return []

    return pronunciations.get(word.lower(), [])


def rhyme_key_from_phones(phones):
    """
    Extract the final stressed vowel and everything after it.
    """
    start_index = None

    # First preference: final primary/secondary stressed vowel.
    for index in range(len(phones) - 1, -1, -1):
        if re.search(r"[12]$", phones[index]):
            start_index = index
            break

    # Fallback: any final vowel-like phone carrying a stress digit.
    if start_index is None:
        for index in range(len(phones) - 1, -1, -1):
            if re.search(r"[012]$", phones[index]):
                start_index = index
                break

    if start_index is None:
        return None

    return tuple(
        re.sub(r"\d", "", phone)
        for phone in phones[start_index:]
    )


def rhyme_keys(word):
    """
    Return all unique rhyme keys for all known pronunciations.
    """
    keys = set()

    for phones in pronunciations_for(word):
        key = rhyme_key_from_phones(phones)

        if key is not None:
            keys.add(key)

    return keys


def rhyme_score(word_a, word_b, allow_identical=False):
    """
    Binary rhyme score:
        1.0 = at least one pronunciation rhyme matches
        0.0 = no known match
    """
    if not word_a or not word_b:
        return 0.0

    word_a = word_a.lower()
    word_b = word_b.lower()

    if not allow_identical and word_a == word_b:
        return 0.0

    keys_a = rhyme_keys(word_a)
    keys_b = rhyme_keys(word_b)

    if not keys_a or not keys_b:
        return 0.0

    return float(bool(keys_a.intersection(keys_b)))


def syllable_count_word(word):
    """
    Estimate syllables from the first available pronunciation.
    """
    entries = pronunciations_for(word)

    if not entries:
        return 0

    phones = entries[0]

    return sum(
        bool(re.search(r"[012]$", phone))
        for phone in phones
    )


def words_in_line(line):
    """
    Extract word tokens for phonetic analysis.
    """
    return re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?", line)


def last_word(line):
    """
    Return the final word in a generated line.
    """
    words = words_in_line(line)

    if not words:
        return None

    return words[-1].lower()


def syllable_count_line(line):
    """
    Sum known word-level syllable estimates.
    """
    return sum(
        syllable_count_word(word)
        for word in words_in_line(line)
    )


# Quick sanity check.
print("Rhyme keys for 'cat':", rhyme_keys("cat"))
print("Rhyme keys for 'hat':", rhyme_keys("hat"))
print("cat / hat rhyme score:", rhyme_score("cat", "hat"))
print("Syllables in 'attack':", syllable_count_word("attack"))



# Inference — Generate candidate lines

## Part 14 — Neural candidate generation

### What this block does
Samples possible poem lines from the fully trained GRU.

### Methodology
Instead of accepting the first sampled line, the system creates several candidates. Each candidate receives:
- a neural language-model score
- later, a rhyme score
- later, a meter/syllable penalty

`top-k` sampling restricts each generation step to relatively likely next tokens, while `temperature` controls randomness.


In [ ]:

"""
TRAINING PROCESS — PART 14: NEURAL CANDIDATE GENERATION

Purpose:
    Generate one candidate poetic line and record its average
    neural log-probability.

Methodology:
    Use temperature-scaled top-k sampling from the trained GRU.
    The language-model score will later be combined with phonetic
    rhyme/meter features.
"""

@torch.no_grad()
def generate_candidate_line(
    model,
    context_ids,
    max_tokens=MAX_TOKENS_PER_LINE,
    temperature=GENERATION_TEMPERATURE,
    top_k=TOP_K
):
    model.eval()

    # Limit historical context so generation remains efficient.
    context_ids = context_ids[-MAX_CONTEXT_TOKENS:]

    context_tensor = torch.tensor(
        [context_ids],
        dtype=torch.long,
        device=DEVICE
    )

    logits, hidden = model(context_tensor)
    next_logits = logits[0, -1]

    generated = []
    total_log_probability = 0.0

    forbidden_tokens = {
        tokenizer.token_to_id["<PAD>"],
        tokenizer.token_to_id["<UNK>"],
        tokenizer.token_to_id["<BOS>"],
        tokenizer.token_to_id["<SONNET>"],
        tokenizer.token_to_id["<DISS>"],
    }

    newline_id = tokenizer.token_to_id["<NL>"]
    eos_id = tokenizer.token_to_id["<EOS>"]

    for _ in range(max_tokens):
        scores = (next_logits / temperature).clone()

        for token_id in forbidden_tokens:
            scores[token_id] = -float("inf")

        if top_k is not None and top_k < scores.numel():
            top_values, top_indices = torch.topk(scores, top_k)

            probabilities = F.softmax(top_values, dim=-1)

            sampled_position = torch.multinomial(
                probabilities,
                num_samples=1
            )

            token_id = top_indices[sampled_position].item()

        else:
            probabilities = F.softmax(scores, dim=-1)

            token_id = torch.multinomial(
                probabilities,
                num_samples=1
            ).item()

        log_probs = F.log_softmax(scores, dim=-1)
        total_log_probability += log_probs[token_id].item()

        if token_id in {newline_id, eos_id}:
            break

        generated.append(token_id)

        next_input = torch.tensor(
            [[token_id]],
            dtype=torch.long,
            device=DEVICE
        )

        logits, hidden = model(next_input, hidden)
        next_logits = logits[0, -1]

    line = tokenizer.decode(generated)

    average_log_probability = (
        total_log_probability / max(1, len(generated))
    )

    return line, generated, average_log_probability



## Part 15 — Combine neural, rhyme, and meter scores

### What this block does
Scores each generated candidate using both learned and rule-based information.

### Methodology

The final score is:

\[
\text{Score}
=
\text{LM Score}
+
\beta(\text{Rhyme})
-
\gamma(\text{Meter Error})
\]

where:
- **LM Score** = average log-probability from the trained GRU
- **Rhyme** = pronunciation-dictionary rhyme match
- **Meter Error** = distance from the target syllable count

This is the key hybrid component that combines all three data sources.


In [ ]:

"""
TRAINING PROCESS — PART 15: HYBRID CANDIDATE SCORING

Purpose:
    Rerank neural candidates using explicit poetry constraints.

Methodology:
    Combine:
        neural plausibility
        + rhyme reward
        - syllable-count penalty

    The weights can be tuned experimentally.
"""

def candidate_score(
    line,
    language_model_score,
    rhyme_target=None,
    target_syllables=TARGET_SYLLABLES,
    rhyme_weight=2.0,
    meter_weight=0.15
):
    score = language_model_score

    # --------------------------------------------------------------
    # Rhyme component
    # --------------------------------------------------------------
    if rhyme_target is not None:
        target_end_word = last_word(rhyme_target)
        candidate_end_word = last_word(line)

        score += rhyme_weight * rhyme_score(
            target_end_word,
            candidate_end_word
        )

    # --------------------------------------------------------------
    # Meter / syllable component
    # --------------------------------------------------------------
    syllables = syllable_count_line(line)

    # Unknown words can make the estimate zero; in that case
    # avoid applying a misleading penalty.
    if syllables > 0:
        meter_error = abs(syllables - target_syllables)
        score -= meter_weight * meter_error

    return score



## Part 16 — Generate a complete AABB-style diss poem

### What this block does
Generates the full poem.

### Methodology
The default rhyme scheme is **AABB**:
- line 1 establishes rhyme A
- line 2 is reranked to rhyme with line 1
- line 3 establishes rhyme B
- line 4 is reranked to rhyme with line 3
- and so on

For each line, multiple candidates are sampled. The highest-scoring candidate is selected and appended to the context for future lines.


In [ ]:

"""
TRAINING PROCESS — PART 16: COMPLETE POEM GENERATION

Purpose:
    Generate an AABB-style diss poem from a user-supplied topic.

Methodology:
    - Sample multiple neural candidates per line.
    - For even-numbered lines, reward rhyme with the preceding line.
    - Penalize syllable counts far from the target.
    - Select the highest-scoring candidate.
"""

@torch.no_grad()
def generate_diss_poem(
    topic,
    model,
    number_of_lines=8,
    candidates_per_line=CANDIDATES_PER_LINE,
    target_syllables=TARGET_SYLLABLES,
    temperature=GENERATION_TEMPERATURE
):
    model.eval()

    topic = clean_text(topic, lowercase=LOWERCASE)

    bos_id = tokenizer.token_to_id["<BOS>"]
    diss_id = tokenizer.token_to_id["<DISS>"]
    newline_id = tokenizer.token_to_id["<NL>"]

    topic_ids = tokenizer.encode(
        topic,
        add_bos=False,
        add_eos=False
    )

    context_ids = [
        bos_id,
        diss_id,
        *topic_ids,
        newline_id
    ]

    selected_lines = []

    for line_index in range(number_of_lines):
        # In AABB, every second line rhymes with the line before it.
        if line_index % 2 == 1 and selected_lines:
            rhyme_target = selected_lines[-1]
        else:
            rhyme_target = None

        candidates = []

        for _ in range(candidates_per_line):
            candidate_line, candidate_ids, lm_score = generate_candidate_line(
                model=model,
                context_ids=context_ids,
                max_tokens=MAX_TOKENS_PER_LINE,
                temperature=temperature,
                top_k=TOP_K
            )

            if not candidate_line.strip():
                continue

            total_score = candidate_score(
                line=candidate_line,
                language_model_score=lm_score,
                rhyme_target=rhyme_target,
                target_syllables=target_syllables
            )

            candidates.append(
                (total_score, candidate_line, candidate_ids, lm_score)
            )

        if not candidates:
            print("Generation stopped: no non-empty candidates.")
            break

        candidates.sort(key=lambda item: item[0], reverse=True)

        best_score, best_line, best_ids, best_lm_score = candidates[0]

        selected_lines.append(best_line)
        context_ids.extend(best_ids)
        context_ids.append(newline_id)

        print(
            f"Line {line_index + 1}: {best_line}\n"
            f"  total score={best_score:.3f}, "
            f"LM score={best_lm_score:.3f}, "
            f"syllables={syllable_count_line(best_line)}, "
            f"end word={last_word(best_line)}"
        )

    return "\n".join(selected_lines)



## Part 17 — Generate an example poem

### What this block does
Runs the complete trained system on an example topic.

### Methodology
Change the `topic` string to evaluate different prompts. For a formal project evaluation, use the **same fixed set of prompts** across all model variants so comparisons remain fair.


In [ ]:

"""
TRAINING PROCESS — PART 17: EXAMPLE INFERENCE

Purpose:
    Demonstrate end-to-end poem generation.

Methodology:
    Use the trained diss-poem model plus pronunciation-based
    candidate reranking.
"""

topic = "an arrogant rival who always claims to be the best"

poem = generate_diss_poem(
    topic=topic,
    model=model,
    number_of_lines=8,
    candidates_per_line=CANDIDATES_PER_LINE,
    target_syllables=TARGET_SYLLABLES,
    temperature=GENERATION_TEMPERATURE
)

print("\n" + "=" * 70)
print("GENERATED POEM")
print("=" * 70)
print(poem)

(OUTPUT_DIR / "generated_poem.txt").write_text(
    poem,
    encoding="utf-8"
)



# Evaluation

## Part 18 — Rhyme and meter metrics

### What this block does
Calculates two interpretable automatic metrics:

1. **Rhyme accuracy**
   - For AABB generation, checks whether each expected rhyme pair has matching phonetic rhyme keys.
2. **Mean absolute syllable error**
   - Measures how far each line is from the desired target syllable count.

### Methodology
These metrics directly evaluate the contribution of the pronunciation-dictionary stage. They should be complemented with language-model validation loss/perplexity and human evaluation of coherence/style.


In [ ]:

"""
TRAINING PROCESS — PART 18: AUTOMATIC POETRY METRICS

Purpose:
    Quantify rhyme success and syllable consistency.

Methodology:
    Evaluate AABB rhyme pairs using the same phonetic dictionary
    used during reranking, and compute absolute syllable deviation.
"""

def evaluate_rhyme_accuracy(poem):
    """
    Evaluate adjacent pairs in an AABB-style poem:
        lines 1-2, 3-4, 5-6, ...
    """
    lines = [
        line.strip()
        for line in poem.split("\n")
        if line.strip()
    ]

    pair_results = []

    for index in range(0, len(lines) - 1, 2):
        word_a = last_word(lines[index])
        word_b = last_word(lines[index + 1])

        keys_a = rhyme_keys(word_a)
        keys_b = rhyme_keys(word_b)

        # Only score pairs for which both words are represented in
        # the pronunciation dictionary.
        if not keys_a or not keys_b:
            continue

        score = rhyme_score(word_a, word_b)

        pair_results.append({
            "line_pair": f"{index + 1}-{index + 2}",
            "word_a": word_a,
            "word_b": word_b,
            "rhymes": bool(score),
        })

    if not pair_results:
        accuracy = float("nan")
    else:
        accuracy = sum(item["rhymes"] for item in pair_results) / len(pair_results)

    return accuracy, pair_results


def evaluate_syllable_error(poem, target_syllables=TARGET_SYLLABLES):
    """
    Compute mean absolute deviation from the desired syllable count.
    """
    lines = [
        line.strip()
        for line in poem.split("\n")
        if line.strip()
    ]

    results = []

    for index, line in enumerate(lines, start=1):
        count = syllable_count_line(line)

        if count == 0:
            continue

        error = abs(count - target_syllables)

        results.append({
            "line": index,
            "syllables": count,
            "absolute_error": error,
        })

    if not results:
        mean_error = float("nan")
    else:
        mean_error = sum(item["absolute_error"] for item in results) / len(results)

    return mean_error, results


rhyme_accuracy, rhyme_details = evaluate_rhyme_accuracy(poem)

meter_error, meter_details = evaluate_syllable_error(
    poem,
    target_syllables=TARGET_SYLLABLES
)

print("Rhyme accuracy:", rhyme_accuracy)
print("Mean absolute syllable error:", meter_error)

print("\nRhyme details:")
display(pd.DataFrame(rhyme_details))

print("\nMeter details:")
display(pd.DataFrame(meter_details))



## Part 19 — Recommended experiment / ablation plan

This project becomes much stronger if the group evaluates **which component caused which improvement**.

Use the **same evaluation prompts** for each system:

| System | Sonnet pretraining | Diss fine-tuning | Pronunciation reranking |
|---|---:|---:|---:|
| A: Diss-only baseline | No | Yes | No |
| B: Sonnet → Diss | Yes | Yes | No |
| C: Full hybrid system | Yes | Yes | Yes |

### Suggested metrics

**Language-model quality**
- validation loss
- perplexity

**Poetry quality**
- rhyme accuracy
- mean absolute syllable error

**Human evaluation**
Use the same 15–20 prompts and ask raters to assess:
- coherence
- diss-style strength
- poetic quality
- rhyme quality

### Interpretation
The purpose of the ablation is not merely to show that the final model generates text. It lets the group test whether:
- sonnet pretraining improves poetic form,
- diss fine-tuning improves target style,
- pronunciation-based reranking improves rhyme/meter.



## Part 20 — Optional helper: save experiment metadata

### What this block does
Writes the main settings and final metrics to a JSON file.

### Methodology
Saving experiment metadata makes later report writing and model comparison easier, especially if you run several configurations with different hyperparameters.


In [ ]:

"""
TRAINING PROCESS — PART 20: SAVE EXPERIMENT METADATA

Purpose:
    Save key hyperparameters and current evaluation metrics.

Methodology:
    Record experiment settings in a machine-readable file so runs
    can be compared and reproduced later.
"""

experiment_metadata = {
    "seed": SEED,
    "device": str(DEVICE),
    "vocab_size": len(tokenizer),
    "sequence_length": SEQUENCE_LENGTH,
    "batch_size": BATCH_SIZE,
    "embedding_dim": EMBEDDING_DIM,
    "hidden_dim": HIDDEN_DIM,
    "num_layers": NUM_LAYERS,
    "dropout": DROPOUT,
    "sonnet_epochs": SONNET_EPOCHS,
    "diss_epochs": DISS_EPOCHS,
    "sonnet_learning_rate": SONNET_LEARNING_RATE,
    "diss_learning_rate": DISS_LEARNING_RATE,
    "target_syllables": TARGET_SYLLABLES,
    "candidates_per_line": CANDIDATES_PER_LINE,
    "pronunciation_source": pronunciation_source,
    "rhyme_accuracy": None if math.isnan(rhyme_accuracy) else rhyme_accuracy,
    "mean_absolute_syllable_error": None if math.isnan(meter_error) else meter_error,
}

metadata_path = OUTPUT_DIR / "experiment_metadata.json"

with metadata_path.open("w", encoding="utf-8") as f:
    json.dump(experiment_metadata, f, indent=2)

print("Saved experiment metadata to:", metadata_path)



## Part 21 — Download trained artifacts from Colab

### What this block does
Packages the final checkpoint, vocabulary, generated example, and experiment metadata into a ZIP file.

### Methodology
This does not affect model training. It simply preserves the artifacts from the temporary Colab runtime so they can be reused for evaluation, demonstration, or report preparation.


In [ ]:

"""
TRAINING PROCESS — PART 21: PACKAGE OUTPUT ARTIFACTS

Purpose:
    Zip model checkpoints and result files for local storage.

Methodology:
    Colab runtimes are temporary, so package important outputs before
    the session ends.
"""

import shutil
from google.colab import files

archive_base = Path("/content/diss_poem_artifacts")

archive_path = shutil.make_archive(
    str(archive_base),
    "zip",
    root_dir=PROJECT_DIR
)

print("Created:", archive_path)

# Uncomment the next line when you want Colab to start the download.
# files.download(archive_path)



# Summary of the methodology

This notebook implements the following training process:

### 1. Preprocessing
- Load sonnet and diss corpora separately.
- Normalize whitespace while preserving line breaks.
- Save `*_cleaned` datasets in `data/cleaned/` and reuse them on later runs.
- Split complete samples into train/validation sets before token-window creation.
- Build one shared vocabulary.

### 2. Stage 1 — Sonnet pretraining
- Train a word-level GRU using next-token prediction.
- Learn general poetic sequencing and line structure.

### 3. Stage 2 — Diss fine-tuning
- Continue training the **same weights** on the diss corpus.
- Use a lower learning rate to specialize the model while retaining poetic knowledge.

### 4. Stage 3 — Pronunciation constraints
- Load CSUdict/CMUdict-style pronunciations.
- Extract rhyme endings and syllable estimates.
- Do **not** train the dictionary as ordinary language-model text.

### 5. Hybrid generation
- Generate multiple neural candidate lines.
- Combine neural likelihood, rhyme reward, and syllable penalty.
- Keep the highest-scoring candidate.

### 6. Evaluation
- Neural validation loss/perplexity
- Rhyme accuracy
- Mean syllable error
- Recommended human evaluation and ablation experiments

This structure makes each dataset's role explicit and gives the project clear experimental comparisons rather than treating all three sources as interchangeable training text.
